![Checkars](https://raw.githubusercontent.com/juliandnl/redi_ss20/master/checkars/grafik.png)

In [2]:
!pip install pandas openpyxl xlrd gdown -q

import os
import pandas as pd
import zipfile
import re
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

FOLDER_ID = "1u4ngjOEEZYW03GBIVw_OZ0fUvwr0DXpc"

Mounted at /content/drive


In [ ]:
# %% [markdown]
# # Smart World Bank Data Loader
# Profiles data first, then downloads only what you select

# %% [code]
# Install required packages
!pip install pandas openpyxl xlrd gdown -q

# %% [code]
import os
import pandas as pd
import zipfile
import io
import re
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

# Mount Drive
drive.mount('/content/drive')

# Folder ID from your link
FOLDER_ID = "1u4ngjOEEZYW03GBIVw_OZ0fUvwr0DXpc"

# %% [code]
# STEP 1: GET FILE LIST WITHOUT DOWNLOADING (FAST!)
print("📋 Scanning folder for files (no download yet)...")
!gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O /content/temp_listing --remaining-ok --quiet

# Find the actual folder
temp_folder = "/content/temp_listing"
for root, dirs, files in os.walk(temp_folder):
    if files:
        folder_path = root
        all_files = files
        break

print(f"✅ Found {len(all_files)} data files\n")

# %% [code]
# STEP 2: DEFINED THEMES (Based on World Bank naming conventions)
THEMES = {
    '🌾 Agriculture': {
        'prefixes': ['AG'],
        'description': 'Agricultural land, irrigation, crop production',
        'files': []
    },
    '⚡ Energy': {
        'prefixes': ['EG'],
        'description': 'Electricity access, renewable energy, energy use',
        'files': []
    },
    '💧 Water & Environment': {
        'prefixes': ['ER', 'EN'],
        'description': 'Water resources, sanitation, environmental indicators',
        'files': []
    },
    '💰 Economy & Finance': {
        'prefixes': ['NY', 'FR', 'FS', 'GB'],
        'description': 'GDP, debt, financial stability, government spending',
        'files': []
    },
    '🏥 Health & Mortality': {
        'prefixes': ['SH', 'SP'],
        'description': 'Mortality rates, health indicators, population',
        'files': []
    },
    '📚 Education': {
        'prefixes': ['SE'],
        'description': 'Literacy, school completion, education spending',
        'files': []
    },
    '📡 Infrastructure & Technology': {
        'prefixes': ['IT', 'IE'],
        'description': 'Internet, phones, infrastructure access',
        'files': []
    },
    '📊 Poverty & Inequality': {
        'prefixes': ['SI'],
        'description': 'Gini index, poverty rates, inequality measures',
        'files': []
    },
    '🔬 Other Indicators': {
        'prefixes': ['HD', 'SL', 'DT'],
        'description': 'Human capital, unemployment, external debt',
        'files': []
    }
}

# %% [code]
# STEP 3: CATEGORIZE FILES INTO THEMES
print("🏷️  Categorizing files into themes...")
print("="*60)

for file in all_files:
    # Extract indicator code from filename
    match = re.search(r'API_([A-Z\.]+)_', file)
    if match:
        code = match.group(1)
        prefix = code.split('.')[0] if '.' in code else code[:2]

        # Find theme
        categorized = False
        for theme, info in THEMES.items():
            if prefix in info['prefixes']:
                file_size = 0  # Will update after download
                info['files'].append({
                    'name': file,
                    'code': code,
                    'size_known': False,
                    'downloaded': False
                })
                categorized = True
                break

        if not categorized:
            THEMES['🔬 Other Indicators']['files'].append({
                'name': file,
                'code': code,
                'size_known': False,
                'downloaded': False
            })
    else:
        # Files without standard naming
        THEMES['🔬 Other Indicators']['files'].append({
            'name': file,
            'code': 'unknown',
            'size_known': False,
            'downloaded': False
        })

# Display categorized files
for theme, info in THEMES.items():
    if info['files']:
        print(f"\n📁 {theme} ({len(info['files'])} files)")
        print(f"   {info['description']}")
        print(f"   First few: {', '.join([f['code'] for f in info['files'][:3]])}")

# %% [code]
# STEP 4: INTERACTIVE THEME SELECTION
print("\n" + "="*60)
print("🎯 SELECT DATA THEMES TO DOWNLOAD")
print("="*60)

theme_list = list(THEMES.keys())
for i, theme in enumerate(theme_list, 1):
    file_count = len(THEMES[theme]['files'])
    print(f"{i:2}. {theme} ({file_count} datasets)")

print(f"{len(theme_list)+1}. Load ALL themes (everything)")

choice = input(f"\n📥 Enter theme numbers (e.g., 1,3,5) or '{len(theme_list)+1}' for all: ")

selected_themes = []
if str(choice) == str(len(theme_list)+1):
    selected_themes = theme_list
else:
    indices = [int(x.strip()) - 1 for x in choice.split(',')]
    selected_themes = [theme_list[i] for i in indices if 0 <= i < len(theme_list)]

# Collect selected files
selected_files = []
for theme in selected_themes:
    for file_info in THEMES[theme]['files']:
        selected_files.append(file_info['name'])

print(f"\n✅ Selected {len(selected_files)} files to download")
print(f"   Total estimated size: ~{len(selected_files) * 0.05:.1f} MB (varies by file)")

# %% [code]
# STEP 5: DOWNLOAD ONLY SELECTED FILES
print("\n" + "="*60)
print("📥 DOWNLOADING SELECTED FILES")
print("="*60)

downloaded_data = {}

for filename in selected_files:
    print(f"\n📄 Downloading: {filename}")

    # Use gdown with file search (need to get file ID first)
    # Alternative: Since we have folder access, download specific files
    try:
        # Download to temp location
        !gdown "https://drive.google.com/drive/folders/{FOLDER_ID}" -O /content/download_temp --remaining-ok --quiet

        # Find the downloaded file
        for root, dirs, files_in_dl in os.walk('/content/download_temp'):
            if filename in files_in_dl:
                file_path = os.path.join(root, filename)

                # Load the data
                with zipfile.ZipFile(file_path, 'r') as z:
                    csv_file = next((f for f in z.namelist() if f.endswith('.csv')), None)
                    if csv_file:
                        with z.open(csv_file) as f:
                            df = pd.read_csv(f, encoding='latin1')

                            # Clean and process
                            print(f"   ✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

                            # Skip World Bank metadata rows (first 4 rows often contain notes)
                            if 'Country Name' in df.columns:
                                df_clean = df.iloc[4:].reset_index(drop=True)
                                print(f"   🧹 Cleaned: Removed metadata rows")
                            else:
                                df_clean = df

                            # Store with metadata
                            downloaded_data[filename] = {
                                'data': df_clean,
                                'shape': df_clean.shape,
                                'columns': list(df_clean.columns),
                                'file_size_mb': os.path.getsize(file_path) / (1024*1024)
                            }
                break

    except Exception as e:
        print(f"   ❌ Error: {e}")

# Clean up temp folder
!rm -rf /content/download_temp

print(f"\n✅ Successfully downloaded and loaded {len(downloaded_data)} files")

# %% [code]
# STEP 6: DISPLAY STATISTICS FOR EACH DATASET
print("\n" + "="*60)
print("📊 DATA STATISTICS & CLEANING REPORT")
print("="*60)

for filename, info in downloaded_data.items():
    df = info['data']

    print(f"\n📈 {filename}")
    print(f"   Theme: {[t for t in THEMES if any(f['name'] == filename for f in THEMES[t]['files'])][0]}")
    print(f"   Size: {info['file_size_mb']:.2f} MB")
    print(f"   Shape: {info['shape'][0]:,} rows × {info['shape'][1]} columns")

    # Find year columns (common in World Bank data)
    year_cols = [col for col in df.columns if str(col).isdigit() and len(str(col)) == 4]
    if year_cols:
        print(f"   📅 Time span: {min(year_cols)} - {max(year_cols)} ({len(year_cols)} years)")

    # Data quality stats
    missing_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
    print(f"   🔍 Missing data: {missing_pct:.1f}%")

    # Show sample columns
    print(f"   📋 Key columns: {', '.join(df.columns[:8])}")
    if len(df.columns) > 8:
        print(f"      ... and {len(df.columns)-8} more columns")

    # Show preview
    print(f"\n   📄 Preview (first 3 rows):")
    print(df.head(3).to_string(max_cols=8))
    print("-" * 50)

# %% [code]
# STEP 7: MAKE DATA ACCESSIBLE FOR ANALYSIS
print("\n" + "="*60)
print("✅ ANALYSIS READY")
print("="*60)

# Create simplified access
for filename, info in downloaded_data.items():
    # Create variable name from filename
    var_name = re.sub(r'[^a-zA-Z0-9]', '_', filename[:30])
    globals()[var_name] = info['data']
    print(f"📊 {var_name} = {filename}")

print(f"\n💡 Access any dataset with its variable name above")
print(f"💡 Or use: downloaded_data['{list(downloaded_data.keys())[0]}']['data']")

# Create combined summary
summary_df = pd.DataFrame([
    {
        'Filename': filename,
        'Rows': info['shape'][0],
        'Columns': info['shape'][1],
        'Size_MB': info['file_size_mb'],
        'Missing_Pct': (info['data'].isnull().sum().sum() / (info['shape'][0] * info['shape'][1])) * 100
    }
    for filename, info in downloaded_data.items()
])

print("\n📊 SUMMARY TABLE:")
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('/content/data_summary.csv', index=False)
print("\n💾 Summary saved to: /content/data_summary.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📋 Scanning folder for files (no download yet)...
Traceback (most recent call last):
  File "/usr/local/bin/gdown", line 10, in <module>
  File "/usr/local/lib/python3.12/dist-packages/gdown/__main__.py", line 157, in main
    download_folder(
  File "/usr/local/lib/python3.12/dist-packages/gdown/download_folder.py", line 328, in download_folder
    local_path = download(
                 ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 208, in download
    res = sess.get(url, stream=True, verify=verify)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/requests/sessions.py", line 602, in get
    return self.request("GET", url, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/requests/sessions.py", line 589, in request
    

In [ ]:
from google.colab import drive
drive.mount('/content/drive')